# 03 — Zero-shot MedSAM2 baseline

Before any fine-tuning, evaluate how well the pretrained MedSAM2 segments synthetic CFRP slices given a bounding-box prompt derived from the GT mask of the middle slice. This establishes the floor that our PEFT runs should clear.

Metrics:
- **2D Dice** per slice (bbox prompt per slice, no memory)
- **3D Dice** over the whole volume
- **Fibre continuity** ratio — how well long Z-directed fibres are preserved

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from cfrp_medsam2.model import SegModel, ModelConfig
from cfrp_medsam2.data import NpzVolume, mask_to_bbox
from cfrp_medsam2.eval import summarize
from cfrp_medsam2.viz import slice_triptych

REPO = Path('..').resolve()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

In [ ]:
cfg = ModelConfig(
    backend='medsam2',
    checkpoint=str(REPO / 'checkpoints' / 'sam2.1_hiera_tiny.pt'),
    image_size=512,
    device=device,
)
model = SegModel(cfg)
model.eval()
print('backend =', model.backend)

## Run zero-shot inference on the validation volume

In [ ]:
from skimage.transform import resize as sk_resize

vol = NpzVolume.load(REPO / 'data' / 'processed' / 'toy' / 'val_00.npz')
Z, H, W = vol.imgs.shape

# Resize to 512 for MedSAM2.
S = 512
imgs_s = np.stack([sk_resize(s, (S, S), order=1, preserve_range=True, anti_aliasing=True).astype(np.uint8) for s in vol.imgs])
gts_s = np.stack([sk_resize(s, (S, S), order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8) for s in vol.gts])
print('resized volume', imgs_s.shape, 'gt fibre voxels', int(gts_s.sum()))

preds = np.zeros_like(gts_s)
with torch.no_grad():
    for z in range(Z):
        rgb = np.repeat(imgs_s[z][None], 3, axis=0).astype(np.float32) / 255.0
        img = torch.from_numpy(rgb).unsqueeze(0).to(device)
        # Box from the GT of this slice (zero-shot uses GT box; training-time uses jittered box too).
        bb = mask_to_bbox(gts_s[z])
        if bb is None:
            continue
        box = torch.tensor(bb, dtype=torch.float32, device=device)
        logits = model.forward_slice(img, boxes=[box])
        preds[z] = (torch.sigmoid(logits[0]).cpu().numpy() > 0.5).astype(np.uint8)

metrics = summarize(preds, gts_s)
print('zero-shot metrics:', metrics)

In [ ]:
# Qualitative — middle slice
z = Z // 2
_ = slice_triptych(imgs_s[z], gts_s[z], preds[z])

## Save baseline numbers so the ablation notebook can pick them up

In [ ]:
import json
out = REPO / 'logs' / 'zero_shot_metrics.json'
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    json.dump(metrics, f, indent=2)
print('wrote', out)